# 10a -- MTMC Stages 0-2: Frame Extraction + Detection + Primary + Secondary ReID Features

**Run once (or when tracking/ReID config changes). ~110-130 min on P100.**

| Stage | What | Time |
|---|---|---|
| 0 | Frame extraction (10 fps, 6 cameras) | ~20 min |
| 1 | YOLO detection + BotSort tracking | ~45 min |
| 2 | TransReID 768D + fast-reid SBS R50-IBN 2048D -> PCA 384D features | ~30 min |

After this runs, its output (`checkpoint.tar.gz`) is used by **10b** -> **10c**.
Attach `mtmc-weights` via **Add Data -> Your Datasets -> mtmc-weights**.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _match = re.search(r"(\d+)\.(\d+)", _cap)
        if _match:
            _major, _minor = _match.groups()
            _sm = int(_major) * 10 + int(_minor)
            if _sm < 70:
                print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])
                print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Keep Kaggle's preinstalled torch/torchvision/numpy stack intact.
pip("--no-deps", "ultralytics")
pip("filterpy", "ftfy", "lapx")
pip("--no-deps", "boxmot==11.0.3")

try:
    import torchreid; print("torchreid ok")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss; print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-gpu")
    except Exception:
        pip("faiss-cpu")

pip("timm", "motmetrics")
pip("loguru", "omegaconf", "rich", "networkx>=3.1", "click")

# SAM2 - non-torch deps first, then SAM2 itself without deps
pip("hydra-core", "iopath")
pip("--no-deps", "git+https://github.com/facebookresearch/sam2.git")

# Install project in editable mode (no deps - all handled above)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
print("✓ All dependencies installed")

In [ ]:
FAILED = []
_checks = [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("cv2", "cv2"),
    ("loguru", "loguru"),
    ("omegaconf", "omegaconf"),
    ("networkx", "networkx"),
    ("sklearn", "sklearn"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sam2", "sam2"),
]
for label, mod in _checks:
    try:
        __import__(mod)
        print(f"  ✓ {label}")
    except ImportError as exc:
        print(f"  ✗ {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
print("✓ All required modules importable")

## 2. Mount Model Weights
Model weights dataset (`mtmc-weights`) must be attached.

In [ ]:
WEIGHTS_INPUT = Path("/kaggle/input/mtmc-weights")
assert WEIGHTS_INPUT.exists(), (
    "Dataset 'mtmc-weights' not found.",
    f"  Expected: {WEIGHTS_INPUT}",
    "  Attach via: Add Data -> Your Datasets -> mtmc-weights",
)

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)
print(f"Copying models/ from {WEIGHTS_INPUT} (~750 MB) ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

# --- Debug: show what was copied ---
print("Contents of models/ after copy:")
for f in sorted(MODELS_DST.rglob("*"))[:30]:
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

# --- Extract any zip files (--dir-mode zip uploads) ---
import zipfile
for zf in sorted(MODELS_DST.rglob("*.zip")):
    print(f"Extracting {zf.relative_to(MODELS_DST)} ...")
    with zipfile.ZipFile(zf) as z:
        z.extractall(zf.parent)
    zf.unlink()

# --- Handle flat structure: move root-level model files into subdirs ---
for subdir in ["detection", "reid", "tracker"]:
    (MODELS_DST / subdir).mkdir(exist_ok=True)

for f in list(MODELS_DST.glob("*.pt")):
    if "yolo" in f.name.lower():
        shutil.move(str(f), str(MODELS_DST / "detection" / f.name))
    elif "osnet" in f.name.lower():
        shutil.move(str(f), str(MODELS_DST / "tracker" / f.name))
for f in list(MODELS_DST.glob("*.pth")):
    shutil.move(str(f), str(MODELS_DST / "reid" / f.name))
for f in list(MODELS_DST.glob("*.pkl")):
    shutil.move(str(f), str(MODELS_DST / "reid" / f.name))
for f in list(MODELS_DST.glob("*.json")):
    if f.name != "dataset-metadata.json":
        shutil.move(str(f), str(MODELS_DST / "reid" / f.name))

print("Final models/ structure:")
for f in sorted(MODELS_DST.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

ESSENTIAL = [
    "models/detection/yolo26m.pt",
    "models/reid/transreid_cityflowv2_best.pth",
    "models/tracker/osnet_x0_25_msmt17.pt",
]
missing = [p for p in ESSENTIAL if not (PROJECT / p).exists()]
if missing:
    for missing_path in missing:
        print(f"  MISSING: {missing_path}")
    raise FileNotFoundError("Fix missing weights before continuing.")
print("✓ All essential baseline weights present")

## 2b. Vehicle ReID Models

Use the baseline CityFlowV2 TransReID checkpoint from `mtmc-weights`.
Download the fast-reid SBS R50-IBN VeRi-776 checkpoint directly from GitHub releases for `vehicle2`.
`vehicle3` stays optional and is enabled only when the 09o EVA02 checkpoint is present as a Kaggle input.

In [ ]:
# Use fine-tuned CityFlowV2 weights from 09n training
FASTREID_FINETUNED_SRC = Path("/kaggle/input/09n-fastreid-r50-finetune-cityflowv2/fastreid_r50_ibn_cityflowv2_final.pth")
FASTREID_SECONDARY_PATH = PROJECT / "models" / "reid" / "fastreid_r50_ibn_cityflowv2.pth"

if FASTREID_FINETUNED_SRC.exists():
    import shutil
    shutil.copy2(FASTREID_FINETUNED_SRC, FASTREID_SECONDARY_PATH)
    print(f"Copied fine-tuned R50-IBN weights: {FASTREID_SECONDARY_PATH}")
else:
    # Fallback: download VeRi-776 pretrained (for testing without 09n)
    FASTREID_SECONDARY_URL = "https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth"
    import urllib.request
    urllib.request.urlretrieve(FASTREID_SECONDARY_URL, str(FASTREID_SECONDARY_PATH))
    print(f"Downloaded VeRi-776 pretrained R50-IBN: {FASTREID_SECONDARY_PATH}")

# Optional vehicle3 EVA02 weights from 09o kernel output.
# Requires the 09o EVA02 training kernel to be added as a Kaggle input.
EVA02_FINETUNED_SRC = Path("/kaggle/input/09o-eva02-vit-reid-cityflowv2/eva02_vit_cityflowv2_final.pth")
EVA02_TERTIARY_PATH = PROJECT / "models" / "reid" / "eva02_vit_cityflowv2.pth"
EVA02_AVAILABLE = False

if EVA02_FINETUNED_SRC.exists():
    import shutil
    shutil.copy2(EVA02_FINETUNED_SRC, EVA02_TERTIARY_PATH)
    EVA02_AVAILABLE = True
    print(f"Copied EVA02 vehicle3 weights: {EVA02_TERTIARY_PATH}")
else:
    print("Warning: EVA02 vehicle3 weights not found at /kaggle/input/09o-eva02-vit-reid-cityflowv2/eva02_vit_cityflowv2_final.pth; vehicle3 will stay disabled")

## 3. Mount CityFlowV2 Dataset

CityFlowV2 is attached as a Kaggle dataset. We symlink the nested
`train/S01/c001/` structure into the flat `S01_c001/` layout the pipeline expects.

In [ ]:
import re as _re, shutil as _shutil

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = _shutil.disk_usage(mount)
    print(f"{mount:20s}  {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

# --- Locate the Kaggle-mounted CityFlowV2 dataset ---
_CANDIDATE_MOUNTS = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = None
for _p in _CANDIDATE_MOUNTS:
    if _p.exists():
        CITYFLOW_INPUT = _p
        break
assert CITYFLOW_INPUT is not None, (
    "CityFlowV2 dataset not found.\n"
    "  Attach via: Add Data -> Datasets -> search 'data-aicity-2023-track-2'"
)
print(f"\u2713 CityFlowV2 dataset at {CITYFLOW_INPUT}")

# --- Flatten into data/raw/cityflowv2/S01_c001/ layout ---
TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)

DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists(): shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)
    print(f"\u2713 data/raw \u2192 {TMP_DATA}")
else:
    print(f"data/raw already symlinked -> {DATA_RAW_PARENT.resolve()}")

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Map: train/S01/c001 -> S01_c001, validation/S02/c006 -> S02_c006
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"  # e.g. S01_c001
            flat_dir = DATA_RAW / flat_name
            if flat_dir.exists():
                continue
            # Symlink the entire camera dir (read-only mount is fine for inference)
            flat_dir.symlink_to(cam_dir)

CAM_RE = _re.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(d.name for d in DATA_RAW.iterdir() if d.is_dir() and CAM_RE.match(d.name))
print(f"\u2713 CityFlowV2 ready: {len(cams)} cameras")
for c in cams: print(f"  {c}")
print(f"\nDataset path: {DATA_RAW}")

## 3b. Generate ROI Masks

Generate per-camera ROI masks from video using background subtraction.
These masks eliminate non-road detections (buildings, parked cars) and are
the single biggest factor in reducing false positives.

In [ ]:
os.chdir(str(PROJECT))
print("Generating ROI masks ...")
r = subprocess.run(
    [sys.executable, "scripts/generate_roi_masks.py",
     "--data-dir", str(DATA_RAW),
     "--n-samples", "200"],
    cwd=str(PROJECT))
if r.returncode != 0:
    print("WARNING: ROI mask generation failed (non-fatal, will proceed without masks)")
else:
    # Verify masks were created
    masks = list(DATA_RAW.glob("*/roi.jpg"))
    print(f"\u2713 Generated {len(masks)} ROI masks")

## 4. Run Stages 0-2

This cell restores the exact v80 override set that produced MTMC IDF1=0.784.
Do not change the listed stage1/stage2 overrides without re-running downstream association sweeps.

In [ ]:
from datetime import datetime
RUN_NAME = f"run_kaggle_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR  = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_CAMERAS = ['S01_c001', 'S01_c002', 'S01_c003', 'S02_c006', 'S02_c007', 'S02_c008']
print(f"Run  : {RUN_NAME}")
print(f"Cams : {BENCHMARK_CAMERAS}")

In [ ]:
os.chdir(str(PROJECT))
import os
import subprocess, time, sys

BASELINE_WEIGHTS = "models/reid/transreid_cityflowv2_best.pth"
SECONDARY_WEIGHTS = "models/reid/fastreid_r50_ibn_cityflowv2.pth"
TERTIARY_WEIGHTS = "models/reid/eva02_vit_cityflowv2.pth"
print(f"Baseline vehicle model: {BASELINE_WEIGHTS} (exists={os.path.exists(BASELINE_WEIGHTS)})")
print(f"Secondary vehicle2 model: {SECONDARY_WEIGHTS} (exists={os.path.exists(SECONDARY_WEIGHTS)})")
print(f"Tertiary vehicle3 EVA02 model: {TERTIARY_WEIGHTS} (exists={os.path.exists(TERTIARY_WEIGHTS)})")

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "0,1,2",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    # v80-restored Stage 0-2 baseline with SAM2 foreground masking disabled and fast-reid vehicle2 enabled
    "--override", "stage2.crop.samples_per_tracklet=48",
    "--override", "stage2.crop.foreground_masking.enabled=false",
    "--override", "stage2.pca.n_components=384",
    "--override", "stage2.reid.color_augment=false",
    "--override", "stage2.reid.vehicle.concat_patch=true",
    "--override", "stage2.reid.vehicle.input_size=[256,256]",
    "--override", "stage2.reid.vehicle2.enabled=true",
    "--override", "stage2.reid.vehicle2.model_name=fastreid_sbs_r50_ibn",
    "--override", "stage2.reid.vehicle2.weights_path=models/reid/fastreid_r50_ibn_cityflowv2.pth",
    "--override", "stage2.reid.vehicle2.embedding_dim=2048",
    "--override", "stage2.reid.vehicle2.input_size=[256,256]",
    "--override", "stage2.reid.vehicle2.clip_normalization=false",
    "--override", "stage2.reid.vehicle2.save_separate=true",
    "--override", "stage2.multi_query.k=0",
    "--override", "stage2.camera_bn.enabled=false",
    "--override", "stage2.camera_tta.enabled=false",
    "--override", "stage2.power_norm.alpha=0.0",
    "--override", "stage1.interpolation.max_gap=50",
    "--override", "stage1.intra_merge.max_time_gap=40.0",
    "--override", "stage1.tracker.min_hits=2",
]

if EVA02_AVAILABLE:
    cmd += [
        "--override", "stage2.reid.vehicle3.enabled=true",
        "--override", "stage2.reid.vehicle3.model_name=eva02_vit",
        "--override", "stage2.reid.vehicle3.weights_path=models/reid/eva02_vit_cityflowv2.pth",
        "--override", "stage2.reid.vehicle3.embedding_dim=768",
        "--override", "stage2.reid.vehicle3.input_size=[256,256]",
        "--override", "stage2.reid.vehicle3.vit_model=eva02_base_patch16_clip_224.merged2b",
        "--override", "stage2.reid.vehicle3.clip_normalization=true",
        "--override", "stage2.reid.vehicle3.num_cameras=0",
        "--override", "stage2.reid.vehicle3.save_separate=true",
    ]
else:
    print("vehicle3 EVA02 overrides skipped because 09o weights are unavailable")

print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"✗ FAILED after {elapsed/60:.1f} min")
    sys.exit(r.returncode)
print(f"✓ Stages 0-2 done in {elapsed/60:.1f} min")

## 5. Save Checkpoint

Saves stage1 + stage2 outputs + GT annotations to `/kaggle/working/checkpoint.tar.gz`.
If `embeddings_secondary.npy` exists in `stage2`, it is included automatically.
This file becomes the input for **10b**.
Stage0 frame images (~9.6 GB) are **not** included -- downstream stages do not need them.

In [ ]:
    import re as _re2

    CAM_RE2 = _re2.compile(r"^S\d{2}_c\d{3}$")
    checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
    metadata_path   = Path("/kaggle/working/run_metadata.json")
    secondary_embeddings_path = RUN_DIR / "stage2" / "embeddings_secondary.npy"

    with open(metadata_path, "w") as f:
        json.dump({"run_name": RUN_NAME}, f)

    print(f"Building checkpoint for run: {RUN_NAME}")
    if secondary_embeddings_path.exists():
        print(f"  ✓ Secondary embeddings detected: {secondary_embeddings_path}")
    else:
        print("  ⚠ Secondary embeddings not found; checkpoint will contain primary stage2 artifacts only")
    with tarfile.open(str(checkpoint_path), "w:gz") as tar:
        tar.add(str(metadata_path), arcname="run_metadata.json")

        manifest = RUN_DIR / "stage0" / "frames_manifest.json"
        if manifest.exists():
            tar.add(str(manifest), arcname=f"{RUN_NAME}/stage0/frames_manifest.json")
            print("  + stage0/frames_manifest.json")

        for stage in ["stage1", "stage2"]:
            stage_dir = RUN_DIR / stage
            if stage_dir.exists():
                n = 0
                for fpath in stage_dir.rglob("*"):
                    if fpath.is_file():
                        tar.add(str(fpath), arcname=f"{RUN_NAME}/{stage}/{fpath.relative_to(stage_dir)}")
                        n += 1
                print(f"  + {stage}/ ({n} files)")

        # GT annotation txt files needed by stage5 eval (small text files, not videos)
        gt_count = 0
        for cam_dir in DATA_RAW.iterdir():
            if cam_dir.is_dir() and CAM_RE2.match(cam_dir.name):
                gt_file = cam_dir / "gt" / "gt.txt"
                if gt_file.exists():
                    tar.add(str(gt_file), arcname=f"gt_annotations/{cam_dir.name}/gt/gt.txt")
                    gt_count += 1
        print(f"  + gt_annotations/ ({gt_count} GT files)")

    size_mb = checkpoint_path.stat().st_size / 1024**2
    print(f"\n✓ Checkpoint: {checkpoint_path}  ({size_mb:.1f} MB)")
    print("  Next: attach this notebook's output to 10b, then push 10b.")